In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
import json
sys.path.append("../../src/experiment_util") 
import experiment_utils
import experiment_constants
sys.path.append("../../src/irl") 
import data_corruption_utils

In [ ]:
with open("../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

In [ ]:

sampled_matched_df = pd.read_pickle(dataframes_path + "/sampled_matched_perturbed_df_final.pkl")

In [ ]:

actions = [
    "Wait reply",
    "New thread",
    "Root comment",
    "Reply comment (agree)",
    "Reply comment (neut)",
    "Reply comment (disagree)"]


# plotting functions

In [ ]:
def plot_policy_radar_plot_multi(dfs, labels, title=""):
    """
    Plot multiple policy radar plots on the same chart.

    Parameters
    ----------
    dfs : list of pd.DataFrame or np.ndarray
        Each dataframe/array contains rows of policies with same number of actions.
    labels : list of str
        Labels corresponding to each dataframe.
    title : str
        Title of the plot.
    """
    actions = [
        "Wait reply",
        "New thread",
        "Root comment",
        "Reply comment (agree)",
        "Reply comment (neut)",
        "Reply comment (disagree)"
    ]
    num_vars = len(actions)

    def compute_stats(data):
        return {
            "min": np.min(data, axis=0),
            "q1": np.percentile(data, 25, axis=0),
            "median": np.percentile(data, 50, axis=0),
            "mean": np.mean(data, axis=0),
            "q3": np.percentile(data, 75, axis=0),
            "max": np.max(data, axis=0),
        }

    # Angles for radar
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    def close_loop(arr):
        return np.concatenate([arr, [arr[0]]])

    # Precompute stats for each dataset
    stats_all = []
    for df in dfs:
        stats = compute_stats(df)
        for k in stats:
            stats[k] = close_loop(stats[k])
        stats_all.append(stats)

    # Color cycle
    colors = itertools.cycle(plt.cm.tab10.colors)

    # Plot
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    for stats, label, color in zip(stats_all, labels, colors):
        # ax.fill(angles, stats["q1"], alpha=0.05, color=color)
        # ax.fill_between(angles, stats["q1"], stats["q3"], alpha=0.15, color=color)
        # ax.plot(angles, stats["min"], color=color, linestyle="--", linewidth=1, alpha=0.7)
        # ax.plot(angles, stats["max"], color=color, linestyle="--", linewidth=1, alpha=0.7)
        ax.plot(angles, stats["median"], color=color, linewidth=2, label=f"{label} ")
        # ax.plot(angles, stats["mean"], color=color, linewidth=2, linestyle=":", label=f"{label} Mean")

    # Axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(actions)

    ax.set_ylim(0, 1.0)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.2))

    plt.title(title)
    plt.tight_layout()
    # plt.savefig(f"{title}.pdf", bbox_inches='tight')
    plt.show()

def plot_policy_radar_plot_multi(dfs, labels, title=""):
    """
    Plot multiple policy radar plots on the same chart.

    Parameters
    ----------
    dfs : list of pd.DataFrame or np.ndarray
        Each dataframe/array contains rows of policies with same number of actions.
    labels : list of str
        Labels corresponding to each dataframe.
    title : str
        Title of the plot.
    """
    actions = [
        "Wait reply",
        "New thread",
        "Root comment",
        "Reply comment (agree)",
        "Reply comment (neut)",
        "Reply comment (disagree)"
    ]
    num_vars = len(actions)

    def compute_stats(data):
        return {
            "min": np.min(data, axis=0),
            "q1": np.percentile(data, 25, axis=0),
            "median": np.percentile(data, 50, axis=0),
            "mean": np.mean(data, axis=0),
            "q3": np.percentile(data, 75, axis=0),
            "max": np.max(data, axis=0),
        }

    # Angles for radar
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    def close_loop(arr):
        return np.concatenate([arr, [arr[0]]])

    # Precompute stats for each dataset
    stats_all = []
    for df in dfs:
        stats = compute_stats(df)
        for k in stats:
            stats[k] = close_loop(stats[k])
        stats_all.append(stats)

    # Color cycle
    colors = itertools.cycle(plt.cm.tab10.colors)

    # Plot
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    for stats, label, color in zip(stats_all, labels, colors):
        ax.fill(angles, stats["q1"], alpha=0.05, color=color)
        ax.fill_between(angles, stats["q1"], stats["q3"], alpha=0.15, color=color)
        # ax.plot(angles, stats["min"], color=color, linestyle="--", linewidth=1, alpha=0.7)
        # ax.plot(angles, stats["max"], color=color, linestyle="--", linewidth=1, alpha=0.7)sampled_non_trolls_df
        ax.plot(angles, stats["median"], color=color, linewidth=2, label=f"{label} Median")
        ax.plot(angles, stats["mean"], color=color, linewidth=2, linestyle=":", label=f"{label} Mean")

    # Axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(actions)

    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.2))

    plt.title(title)
    plt.tight_layout()
    # plt.savefig(f"{title}.pdf", bbox_inches='tight')
    plt.show()

# Visualing policies

In [ ]:
subset_users = sampled_matched_df[sampled_matched_df.user.isin(sampled_matched_df.user.unique())]
rus_user_df = subset_users[subset_users.russian == 1].copy()
non_rus_user_df = subset_users[subset_users.russian == 0].copy()

trolls_df = sampled_matched_df[(sampled_matched_df.russian == 1) & (sampled_matched_df.run == 0) & (sampled_matched_df.perturb_percent == 0.0) ].copy().reset_index(drop=True)
sampled_non_trolls_df = sampled_matched_df[(sampled_matched_df.russian == 0) & (sampled_matched_df.run == 0) & (sampled_matched_df.perturb_percent == 0.0) ].copy().reset_index(drop=True)


In [ ]:
# looking at policies
flattened_policies_list = []

for p in trolls_df.gail_policy.values:
    flattened_policies_list.append(np.mean(p,0))
trolls_df["flattened_policies"] = flattened_policies_list

flattened_policies_list = []
for p in sampled_non_trolls_df.gail_policy.values:
    flattened_policies_list.append(np.mean(p,0))
sampled_non_trolls_df["flattened_policies"] = flattened_policies_list


troll_policies = np.vstack(trolls_df.flattened_policies.values)
non_troll_policies = np.vstack(sampled_non_trolls_df.flattened_policies.values)
plot_policy_radar_plot_multi([troll_policies, non_troll_policies], ["Trolls", "Non-Trolls"], "GAIL Trolls vs Non-trolls")

## clustering

In [ ]:
flattened_policies_list = []

for p in trolls_df.gail_policy.values:
    flattened_policies_list.append(np.mean(p,0))

trolls_df["flattened_policies"] = flattened_policies_list
flattened_policies = trolls_df.flattened_policies.values

X = np.vstack(flattened_policies)

n_clusters = 3
cluster_labels = range(n_clusters)

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(X)
# set clusters
trolls_df["cluster"] = clusters

centroids = kmeans.cluster_centers_
_, cluster_counts = np.unique(clusters, return_counts=True)

for c in range(len(centroids)):
    print("cluster", c, cluster_counts[c])
    cent = centroids[c]
    
    df = pd.DataFrame(cent.reshape(1,6), columns=actions)
    plt.figure(figsize=(3, 1))
    sns.heatmap(df, cmap='Greens', annot=True, fmt=".2f", linewidths=.5)
    plt.title("")
    plt.xlabel('Actions')
    plt.ylabel('State')
    plt.show()


In [ ]:
cluster_dfs = []

for c in [2,1,0]:#range(len(centroids)):
    print("cluster", c, cluster_counts[c])
    flattened_policies = trolls_df[trolls_df.cluster == c].flattened_policies.values
    flattened_policies = np.vstack(flattened_policies)
    cluster_dfs.append(flattened_policies)

plot_policy_radar_plot_multi(cluster_dfs,["Cluster 1","Cluster 2","Cluster 3"],"GAIL troll clusters")